In [12]:
import pandas as pd
import os
import numpy as np

group_cols = ["User", "Model", "Device"]

unmerged_acc = pd.read_csv('unmerged_acc.csv')
unmerged_gyro = pd.read_csv('unmerged_gyro.csv')
merged_df = pd.read_csv('merged_data.csv')

merged_df.head()

,Arrival_Time,Creation_Time,x_x,y_x,z_x,User,Model,Device,gt_x,acc_index,x_y,y_y,z_y,gt_y,gyro_index
0,1424696633911,207817520343346,-9.160782,-3.759674,1.396469,a,lgwatch,lgwatch_1,stand,42578,0.021973,0.003754,-0.071884,stand,42406
1,1424696633916,207817525287194,-9.198868,-3.788238,1.420273,a,lgwatch,lgwatch_1,stand,42579,0.025177,-0.000519,-0.071884,stand,42407
2,1424696633921,207817530231042,-9.208389,-3.804901,1.439316,a,lgwatch,lgwatch_1,stand,42580,0.030518,0.000549,-0.072952,stand,42408
3,1424696633926,207817535174889,-9.210770,-3.759674,1.446457,a,lgwatch,lgwatch_1,stand,42581,0.031586,0.002686,-0.069748,stand,42409
4,1424696633931,207817540118737,-9.222672,-3.735870,1.377426,a,lgwatch,lgwatch_1,stand,42582,0.033722,-0.000519,-0.067612,stand,42410


In [13]:

print("Unmerged_acc count:", len(unmerged_acc))
print("Unmerged_gyro count:", len(unmerged_gyro))
print("Merged_df count:", len(merged_df))

# 2. Extract unique combinations from merged_df
unique_groups = merged_df[group_cols].drop_duplicates().reset_index(drop=True)

# 3. Calculate row counts per group across datasets
merged_counts = merged_df.groupby(group_cols).size().rename("merged_count")
acc_counts = unmerged_acc.groupby(group_cols).size().rename("unmerged_acc_count")
gyro_counts = unmerged_gyro.groupby(group_cols).size().rename("unmerged_gyro_count")

# 4. Join counts into a single inspection table
group_breakdown = (
    unique_groups
    .set_index(group_cols)
    .join(merged_counts, how='left')
    .join(acc_counts, how='left')
    .join(gyro_counts, how='left')
    .fillna(0)
    .astype(int)
    .reset_index()
)

# 5. Identify incomplete groups (unmerged count > 0 in either dataset)
group_breakdown['is_incomplete'] = (group_breakdown['unmerged_acc_count'] > 0) | (group_breakdown['unmerged_gyro_count'] > 0)

incomplete_groups = group_breakdown[group_breakdown['is_incomplete']]

# 6. Summary checks
print(f"Total groups in merged_df: {len(group_breakdown)}")
print(f"Incomplete groups in merged_df: {len(incomplete_groups)}")

print("\n--- Incomplete Groups Breakdown ---")
print(incomplete_groups)

Unmerged_acc count: 14829873
Unmerged_gyro count: 15364499
Merged_df count: 1773564
Total groups in merged_df: 15
Incomplete groups in merged_df: 15

--- Incomplete Groups Breakdown ---
   User    Model     Device  merged_count  unmerged_acc_count  \
0     a  lgwatch  lgwatch_1        267700              113543   
1     a  lgwatch  lgwatch_2           400               72203   
2     b  lgwatch  lgwatch_1        317533              135682   
3     c  lgwatch  lgwatch_1        274802              116703   
4     c  lgwatch  lgwatch_2          3128               61077   
5     d  lgwatch  lgwatch_1        261027              108849   
6     d  lgwatch  lgwatch_2          2438               29883   
7     e  lgwatch  lgwatch_1        324187              135259   
8     e  lgwatch  lgwatch_2         11961               92922   
9     f  lgwatch  lgwatch_1        306845              138749   
10    f  lgwatch  lgwatch_2           202               72148   
11    g  lgwatch  lgwatch_2       

In [14]:
df_watch_acc = pd.read_csv("../ignore_files/data/full/Watch_accelerometer.csv", index_col=0)
df_phone_acc = pd.read_csv("../ignore_files/data/full/Phones_accelerometer.csv", index_col=0)

df_watch_gyro = pd.read_csv("../ignore_files/data/full/Watch_gyroscope.csv", index_col=0)
df_phone_gyro = pd.read_csv("../ignore_files/data/full/Phones_gyroscope.csv", index_col=0)

# Concat watch and phone dataframes for accelerometer and gyroscope
df_acc = pd.concat([df_watch_acc, df_phone_acc], ignore_index=True)
df_gyro = pd.concat([df_watch_gyro, df_phone_gyro], ignore_index=True)

# 1. Extract unique groups from each DataFrame
acc_groups = df_acc[group_cols].drop_duplicates()
gyro_groups = df_gyro[group_cols].drop_duplicates()

# 2. Combine and drop duplicates to get all distinct groups
all_groups = pd.concat([acc_groups, gyro_groups]).drop_duplicates().reset_index(drop=True)

print(f"Total unique groups across both datasets: {len(all_groups)}")
print(all_groups)

Total unique groups across both datasets: 107
    User       Model        Device
0      a        gear        gear_1
1      a        gear        gear_2
2      a     lgwatch     lgwatch_1
3      a     lgwatch     lgwatch_2
4      b        gear        gear_1
..   ...         ...           ...
102    i      s3mini      s3mini_2
103    i  samsungold  samsungold_1
104    i  samsungold  samsungold_2
105    i        gear        gear_2
106    f          s3          s3_1

[107 rows x 3 columns]
